## Generating statistics to make the script more realistic

In [22]:
import os
import json
import numpy as np
from collections import defaultdict
from typing import Dict, List, Any

In [23]:
def as_list_on_duplicate_keys(ordered_pairs):
    d = {}
    for k, v in ordered_pairs:
        if k in d:
            if isinstance(d[k], list): d[k].append(v)
            else: d[k] = [d[k], v]
        else: d[k] = v
    return d

In [24]:
def calculate_statistics_from_captures(directory_path: str) -> dict:
    """
    Analyzes QUIC capture JSON files to build comprehensive statistical profiles.
    
    Extracts statistics for:
    - Packet sizes (handshake, data, ACKs, control frames, HTTP/3 streams)
    - Delta times (handshake, data transfer, ACKs)
    - ACK frequency patterns
    - Path validation (PATH_CHALLENGE/RESPONSE)
    - Connection migration timing
    
    Args:
        directory_path: Path to directory containing Wireshark JSON exports
        
    Returns:
        Dictionary with statistical profiles (mean, std, sample count)
    """
    raw_stats = defaultdict(lambda: defaultdict(list))
    
    print(f"Analyzing JSON files in: {directory_path}\n")
    
    for filename in os.listdir(directory_path):
        if not filename.endswith('.json'):
            continue
            
        json_file_path = os.path.join(directory_path, filename)
        print(f"  -> Processing: {filename}")
        
        # Load JSON with encoding fallback
        try:
            with open(json_file_path, 'r', encoding='utf-8') as f:
                packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
        except (json.JSONDecodeError, UnicodeDecodeError):
            try:
                with open(json_file_path, 'r', encoding='utf-16') as f:
                    packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)
            except Exception as e:
                print(f"     ... Skipping, could not decode JSON: {e}")
                continue
        
        if not packets:
            continue
        
        # Extract initial connection parameters
        try:
            first_packet_layers = packets[0]['_source']['layers']
            if not ('ip' in first_packet_layers and 'frame' in first_packet_layers):
                continue
                
            initial_ip_client = first_packet_layers['ip']['ip.src']
            initial_ip_server = first_packet_layers['ip']['ip.dst']
            initial_port_client = int(first_packet_layers['udp']['udp.srcport'])
            initial_port_server = int(first_packet_layers['udp']['udp.dstport'])
            last_packet_time = float(first_packet_layers['frame']['frame.time_epoch'])
        except (KeyError, IndexError):
            print(f"     ... Skipping, missing required fields")
            continue
        
        # State tracking
        in_handshake = True
        migrated = False
        handshake_done_seen = False
        client_data_since_server_ack = 0
        server_data_since_client_ack = 0
        
        # Process each packet
        for pkt_index, pkt_data in enumerate(packets):
            layers = pkt_data.get('_source', {}).get('layers', {})
            if not ('ip' in layers and 'frame' in layers):
                continue
            
            # Extract packet metadata
            src_ip = layers['ip']['ip.src']
            dst_ip = layers['ip']['ip.dst']
            src_port = int(layers['udp']['udp.srcport'])
            dst_port = int(layers['udp']['udp.dstport'])
            
            # Detect migration
            if not migrated and \
               (dst_ip == initial_ip_server and dst_port == initial_port_server) and \
               (src_ip != initial_ip_client or src_port != initial_port_client):
                migrated = True
                raw_stats['behavior_counts']['packets_before_migration'].append(pkt_index)
            
            # Calculate timing
            current_time = float(layers['frame']['frame.time_epoch'])
            delta_time = current_time - last_packet_time
            last_packet_time = current_time
            
            # Direction
            is_client_pkt = (src_ip == initial_ip_client and src_port == initial_port_client)
            direction = 'client' if is_client_pkt else 'server'
            
            # Packet length
            pkt_len = int(layers['frame']['frame.len'])
            
            # Extract QUIC layer
            quic_packet_list = layers.get('quic', [])
            if not isinstance(quic_packet_list, list):
                quic_packet_list = [quic_packet_list]
            
            # Process each QUIC packet (can have multiple per UDP packet)
            for quic_packet in quic_packet_list:
                # Get packet type
                packet_type = quic_packet.get('quic.long.packet_type')
                is_long_header = quic_packet.get('quic.header_form') == '1'
                
                # Extract frames
                quic_frames = quic_packet.get('quic.frame', [])
                if not isinstance(quic_frames, list):
                    quic_frames = [quic_frames]
                
                # Analyze frame types
                frame_types = {f.get('quic.frame_type', '0') for f in quic_frames}
                
                # Frame type checks
                has_crypto = '0x0000000000000006' in frame_types
                has_ack = '0x0000000000000002' in frame_types or '0x0000000000000003' in frame_types
                has_stream = any('0x0000000000000008' <= ft <= '0x000000000000000f' for ft in frame_types)
                has_padding = '0x0000000000000000' in frame_types
                has_ping = '0x0000000000000001' in frame_types
                has_connection_close = '0x000000000000001c' in frame_types or '0x000000000000001d' in frame_types
                has_path_challenge = '0x000000000000001a' in frame_types
                has_path_response = '0x000000000000001b' in frame_types
                has_new_connection_id = '0x0000000000000018' in frame_types
                has_handshake_done = '0x000000000000001e' in frame_types
                
                # Detect handshake completion
                if has_handshake_done:
                    handshake_done_seen = True
                    in_handshake = False
                
                # ============================================================
                # HANDSHAKE PHASE STATISTICS
                # ============================================================
                if in_handshake or (is_long_header and has_crypto):
                    # Delta times
                    delta_key = 'handshake_c2s' if is_client_pkt else 'handshake_s2c'
                    raw_stats['delta_times'][delta_key].append(delta_time)
                    
                    # Packet sizes
                    if packet_type == '0' and is_client_pkt:
                        # Initial packet from client
                        raw_stats['packet_sizes']['handshake_initial_client'].append(pkt_len)
                    elif packet_type == '0' and not is_client_pkt:
                        # Initial packet from server
                        raw_stats['packet_sizes']['handshake_initial_server'].append(pkt_len)
                    elif packet_type == '2':
                        # Handshake packet
                        raw_stats['packet_sizes'][f'handshake_handshake_{direction}'].append(pkt_len)
                    elif packet_type == '3':
                        # Retry packet
                        raw_stats['packet_sizes']['handshake_retry'].append(pkt_len)
                    else:
                        # Other handshake packets
                        raw_stats['packet_sizes'][f'handshake_other_{direction}'].append(pkt_len)
                
                # ============================================================
                # 1-RTT PHASE STATISTICS
                # ============================================================
                elif not in_handshake or handshake_done_seen:
                    # PATH_CHALLENGE/RESPONSE (connection migration)
                    if has_path_challenge:
                        raw_stats['packet_sizes']['path_challenge'].append(pkt_len)
                        raw_stats['delta_times']['path_challenge'].append(delta_time)
                    
                    if has_path_response:
                        raw_stats['packet_sizes']['path_response'].append(pkt_len)
                        raw_stats['delta_times']['path_response'].append(delta_time)
                    
                    # ACK-only packets
                    if has_ack and not has_stream and not has_connection_close and not has_path_challenge and not has_path_response:
                        raw_stats['packet_sizes'][f'ack_{direction}'].append(pkt_len)
                        raw_stats['delta_times']['ack_response'].append(delta_time)
                        
                        # Track ACK frequency
                        if is_client_pkt:
                            if server_data_since_client_ack > 0:
                                raw_stats['ack_frequency']['client_sends_ack_after'].append(server_data_since_client_ack)
                            server_data_since_client_ack = 0
                        else:
                            if client_data_since_server_ack > 0:
                                raw_stats['ack_frequency']['server_sends_ack_after'].append(client_data_since_server_ack)
                            client_data_since_server_ack = 0
                    
                    # PING-only packets
                    elif has_ping and not has_stream and not has_connection_close:
                        raw_stats['packet_sizes'][f'ping_{direction}'].append(pkt_len)
                        raw_stats['delta_times'][f'ping_{direction}'].append(delta_time)
                    
                    # CONNECTION_CLOSE packets
                    elif has_connection_close:
                        raw_stats['packet_sizes'][f'close_{direction}'].append(pkt_len)
                        raw_stats['delta_times']['close'].append(delta_time)
                    
                    # STREAM packets (application data)
                    elif has_stream:
                        delta_key = 'client_request' if is_client_pkt else 'server_response'
                        raw_stats['delta_times'][delta_key].append(delta_time)
                        
                        # Track for ACK frequency
                        if is_client_pkt:
                            client_data_since_server_ack += 1
                        else:
                            server_data_since_client_ack += 1
                        
                        # Categorize by stream type (based on stream ID)
                        stream_type_found = False
                        for frame in quic_frames:
                            frame_type = frame.get('quic.frame_type', '0')
                            if '0x0000000000000008' <= frame_type <= '0x000000000000000f':
                                stream_id = frame.get('quic.stream.stream_id')
                                
                                if stream_id is not None:
                                    try:
                                        stream_id = int(stream_id)
                                        
                                        # Determine stream type from stream ID
                                        # Stream ID modulo 4 determines initiator and directionality
                                        if stream_id % 4 == 0:
                                            stream_type_key = 'pkt_size_bidi_client'
                                        elif stream_id % 4 == 1:
                                            stream_type_key = 'pkt_size_bidi_server'
                                        elif stream_id % 4 == 2:
                                            stream_type_key = 'pkt_size_uni_client'
                                        elif stream_id % 4 == 3:
                                            stream_type_key = 'pkt_size_uni_server'
                                        
                                        raw_stats['packet_sizes'][stream_type_key].append(pkt_len)
                                        stream_type_found = True
                                        break  # Use first stream in packet
                                    except (ValueError, TypeError):
                                        continue
                        
                        # Fallback if no stream ID found
                        if not stream_type_found:
                            raw_stats['packet_sizes'][f'stream_data_{direction}'].append(pkt_len)
                    
                    # Mixed packets (have ACK + data)
                    elif has_ack and has_stream:
                        # Already counted in stream category above
                        pass
    
    # ============================================================
    # COMPUTE FINAL STATISTICS
    # ============================================================
    final_stats = defaultdict(dict)
    
    for category, keys in raw_stats.items():
        for key, data_list in keys.items():
            # Filter out invalid values
            if 'ack_frequency' in category:
                data_list = [x for x in data_list if x > 0]
            
            if 'delta_times' in category:
                data_list = [x for x in data_list if x >= 0]
            
            if 'packet_sizes' in category:
                data_list = [x for x in data_list if x > 0]
            
            # Calculate statistics
            if data_list and len(data_list) > 0:
                final_stats[category][key] = {
                    'mean': float(np.mean(data_list)),
                    'std': float(np.std(data_list)),
                    'min': float(np.min(data_list)),
                    'max': float(np.max(data_list)),
                    'median': float(np.median(data_list)),
                    'samples': len(data_list)
                }
            else:
                final_stats[category][key] = {
                    'mean': 0.0,
                    'std': 0.0,
                    'min': 0.0,
                    'max': 0.0,
                    'median': 0.0,
                    'samples': 0
                }
    
    print(f"\n=== Statistics Summary ===")
    print(f"Packet size categories: {len(final_stats.get('packet_sizes', {}))}")
    print(f"Delta time categories: {len(final_stats.get('delta_times', {}))}")
    print(f"ACK frequency metrics: {len(final_stats.get('ack_frequency', {}))}")
    print(f"Behavior counts: {len(final_stats.get('behavior_counts', {}))}")
    
    return dict(final_stats)


def print_statistics_summary(stats: Dict[str, Any]) -> None:
    """Pretty print statistics summary."""
    print("\n" + "="*80)
    print("STATISTICS SUMMARY")
    print("="*80)
    
    for category, metrics in stats.items():
        print(f"\n{category.upper()}:")
        print("-" * 80)
        
        for key, values in sorted(metrics.items()):
            if values['samples'] > 0:
                print(f"  {key:40s} | μ={values['mean']:8.4f}  σ={values['std']:8.4f}  "
                      f"n={values['samples']:4d}  [{values['min']:.2f}, {values['max']:.2f}]")
            else:
                print(f"  {key:40s} | No samples")


In [25]:
json_captures_directory = r"C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures\test"

if not os.path.isdir(json_captures_directory):
    print(f"Error: Directory not found at '{json_captures_directory}'")
    print("Please update the 'json_captures_directory' variable with the correct path.")
else:
    full_stats_profile = calculate_statistics_from_captures(json_captures_directory)
    
    print("\n\n--- Calculated Statistical Profile ---")
    print(json.dumps(full_stats_profile, indent=4))

Analyzing JSON files in: C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures\test

  -> Processing: 1_quiche_capture.json
  -> Processing: 2_quiche_capture.json
  -> Processing: 3172_aioquic_before_fast.json
  -> Processing: 3173_aioquic_before_fast.json
  -> Processing: 3174_aioquic_before_fast.json
  -> Processing: 3175_aioquic_before_fast.json
  -> Processing: 3941_aioquic_before_slow_600.json
  -> Processing: 3942_aioquic_before_slow_600.json
  -> Processing: 3943_aioquic_before_slow_600.json
  -> Processing: 3944_aioquic_before_slow_600.json
  -> Processing: 3_quiche_capture.json
  -> Processing: 4234_aioquic_during_slow_200.json
  -> Processing: 4235_aioquic_during_slow_200.json
  -> Processing: 4236_aioquic_during_slow_200.json
  -> Processing: 4237_aioquic_during_slow_200.json
  -> Processing: 4918_aioquic_during_slow_600.json
  -> Processing: 4919_aioquic_during_slow_600.json
  -> Processing: 4920_aioquic_during_slow_600.json
  -> Processing: 4921_ai

In [26]:
print_statistics_summary(full_stats_profile)


STATISTICS SUMMARY

DELTA_TIMES:
--------------------------------------------------------------------------------
  ack_response                             | μ=  0.0080  σ=  0.0082  n= 119  [0.00, 0.03]
  client_request                           | μ=  0.0049  σ=  0.0079  n=  16  [0.00, 0.02]
  close                                    | μ=  0.2803  σ=  0.4504  n=  29  [0.00, 1.01]
  handshake_c2s                            | μ=  0.0014  σ=  0.0010  n= 215  [0.00, 0.00]
  handshake_s2c                            | μ=  0.0027  σ=  0.0023  n= 102  [0.00, 0.01]
  path_challenge                           | μ=  0.0006  σ=  0.0003  n=  34  [0.00, 0.00]
  path_response                            | μ=  0.0037  σ=  0.0070  n=  34  [0.00, 0.03]
  ping_server                              | μ=  0.0097  σ=  0.0135  n=  15  [0.00, 0.03]
  server_response                          | μ=  0.0269  σ=  0.0519  n= 161  [0.00, 0.15]

PACKET_SIZES:
------------------------------------------------------------

In [27]:
full_stats_profile

{'delta_times': {'handshake_c2s': {'mean': 0.0013973712921142578,
   'std': 0.000967970295872902,
   'min': 0.0,
   'max': 0.003258943557739258,
   'median': 0.001772165298461914,
   'samples': 215},
  'handshake_s2c': {'mean': 0.0026799814373839135,
   'std': 0.002345290535902211,
   'min': 0.00011301040649414062,
   'max': 0.009875059127807617,
   'median': 0.0031070709228515625,
   'samples': 102},
  'server_response': {'mean': 0.026936246741632495,
   'std': 0.051887949764005736,
   'min': 0.0,
   'max': 0.14530014991760254,
   'median': 0.0008790493011474609,
   'samples': 161},
  'ack_response': {'mean': 0.00799737056764234,
   'std': 0.008152223605089283,
   'min': 0.0,
   'max': 0.029800891876220703,
   'median': 0.007986068725585938,
   'samples': 119},
  'path_challenge': {'mean': 0.0005812644958496094,
   'std': 0.0003131434010756422,
   'min': 0.0001990795135498047,
   'max': 0.0015611648559570312,
   'median': 0.0005314350128173828,
   'samples': 34},
  'path_response': {'

#### Generating low level features with the statistics

In [ ]:
import numpy as np
import csv
import pandas as pd
from typing import Dict, List, Tuple, Optional

pd.set_option('display.max_columns', None)  # Show all columns when printing

SIMULATED_MTU = 1350
PATH_VALIDATION_MTU_MIN = 1200  # RFC 9000 minimum for path validation
PATH_VALIDATION_MTU_MAX = 1450  # Allow some variance


def generate_statistically_realistic_features(blueprint: dict, stats: dict) -> list:
    """
    Generates a flexible, statistics-driven sequence of QUIC packets.
    
    This version:
    - Uses statistics for ALL packet sizes (not hardcoded)
    - Adds realistic randomness to delta times based on network conditions
    - Adapts to the high-level blueprint while maintaining protocol correctness
    - Generates varied captures from the same blueprint
    """
    output_packet_features = []
    current_time_msec = 0.0
    current_packet_count = 0
    
    # Track application bytes
    client_app_bytes_sent = 0
    server_app_bytes_sent = 0
    has_migrated = False
    
    # Network jitter simulation (adds realism to delta times)
    base_rtt = blueprint.get('migration_validation_duration_msec', 20.0) / 2.0  # Half RTT
    network_jitter = np.random.uniform(0.5, 1.5)  # Random network conditions
    
    def get_stat(category: str, key: str, fallback_mean: float = 100, 
                 fallback_std: float = 20, strict_positive: bool = False) -> float:
        """
        Safely draw from statistics with fallbacks and bounds.
        """
        if category in stats and key in stats[category] and stats[category][key]['samples'] > 0:
            mean = stats[category][key]['mean']
            std = stats[category][key]['std']
            value = np.random.normal(mean, std)
        else:
            value = np.random.normal(fallback_mean, fallback_std)
        
        # Apply bounds
        if 'delta' in key or 'time' in key:
            return max(0.000001, value) * network_jitter
        elif 'size' in key or 'length' in key:
            min_val = 50 if strict_positive else 40
            return int(max(min_val, min(SIMULATED_MTU, value)))
        elif strict_positive:
            return max(1, value)
        return value
    
    def get_delta_time(phase: str, direction: str = 'both') -> float:
        """
        Get realistic delta time based on phase and direction.
        Adds randomness for network conditions.
        """
        if phase == 'handshake':
            if direction == 'c2s':
                base = get_stat('delta_times', 'handshake_c2s', 0.015, 0.010)
            else:
                base = get_stat('delta_times', 'handshake_s2c', 0.020, 0.015)
        elif phase == 'data':
            if direction == 'c2s':
                base = get_stat('delta_times', 'client_request', 0.010, 0.005)
            else:
                base = get_stat('delta_times', 'server_response', 0.005, 0.003)
        elif phase == 'ack':
            base = get_stat('delta_times', 'ack_response', 0.002, 0.001)
        else:
            base = 0.001
        
        # Add random jitter (10-50% variance)
        jitter = np.random.uniform(0.7, 1.3)
        return max(0.000001, base * jitter)
    
    def get_packet_size(packet_type: str, fallback_mean: int = 100, 
                       fallback_std: int = 20) -> int:
        """
        Get packet size from statistics with protocol-aware fallbacks.
        """
        # Try to get from stats first
        size_key = f'{packet_type}'
        size = get_stat('packet_sizes', size_key, fallback_mean, fallback_std)
        
        # Ensure minimum sizes for specific packet types
        if 'initial' in packet_type.lower():
            return max(1200, int(size))  # Initial packets need to be large
        elif 'path_' in packet_type.lower():
            return int(np.random.uniform(PATH_VALIDATION_MTU_MIN, PATH_VALIDATION_MTU_MAX))
        elif 'ack' in packet_type.lower():
            return max(60, min(150, int(size)))  # ACKs are small
        
        return int(size)
    
    def create_packet(delta: float, length: int, direction: int, header_form: int,
                     **counts) -> dict:
        """
        Create a packet feature dictionary with all fields.
        """
        nonlocal current_packet_count, current_time_msec
        current_packet_count += 1
        current_time_msec += delta
        
        # Default all counts to 0
        packet = {
            'frame_number': current_packet_count,
            'delta_time': round(delta, 6),
            'packet_length': int(length),
            'packet_direction': direction,
            'header_form': header_form,
            'count_initial': 0,
            'count_0rtt': 0,
            'count_handshake': 0,
            'count_1rtt': 0,
            'count_retry': 0,
            'count_vn': 0,
            'count_ack': 0,
            'count_padding': 0,
            'count_connection_close': 0,
            'count_path_challenge': 0,
            'count_path_response': 0,
            'count_new_connection_id': 0,
            'count_retire_cid': 0,
            'count_crypto': 0,
            'count_handshake_done': 0,
            'http3_stream_count': 0,
            'http3_fin_count': 0,
            'stream_length': 0,
            'stream_type_count': 0
        }
        
        # Update with provided counts
        packet.update(counts)
        return packet
    
    # =================================================================
    # PHASE 1: HANDSHAKE
    # =================================================================
    if blueprint.get('retry_occurred', 0) == 1:
        # Retry handshake sequence
        # 1. Client Initial attempt
        output_packet_features.append(create_packet(
            0.0,
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 0, count_1rtt=1
        ))
        
        # 2. Server early response (VN or similar)
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('ack_server', 95, 15),
            1, 0, count_1rtt=1, count_vn=1
        ))
        
        # 3. Client Initial (after seeing VN)
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 1, count_initial=1, count_crypto=1
        ))
        
        # 4. Server Retry
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('handshake_other_server', 137, 20),
            1, 1, count_retry=1
        ))
        
        # 5. Client Initial with retry token
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 1, count_initial=1, count_crypto=1
        ))
        
        # 6. Server Initial + Handshake (large packet)
        num_crypto = np.random.randint(1, 3)  # Variable crypto frames
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('handshake_initial_client', 1248, 50),
            1, 1, 
            count_initial=1, 
            count_handshake=np.random.randint(0, 2),
            count_ack=1, 
            count_crypto=num_crypto
        ))
        
        # 7. Server Handshake continuation (if needed)
        if np.random.random() > 0.3:  # 70% chance of continuation packet
            output_packet_features.append(create_packet(
                get_delta_time('handshake', 's2c'),
                get_packet_size('handshake_other_server', 517, 100),
                1, 1, count_handshake=1, count_crypto=1
            ))
        
        # 8. Client Handshake completion
        num_packet_types = np.random.randint(2, 4)  # Variable packet type mixing
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_other_client', 1398, 100),
            0, 1,
            count_initial=np.random.randint(0, 2),
            count_handshake=1,
            count_1rtt=np.random.randint(0, 2),
            count_ack=np.random.randint(1, 3),
            count_padding=np.random.randint(0, 2),
            count_new_connection_id=1,
            count_crypto=1
        ))
    else:
        # Standard handshake (no retry)
        # 1. Client Initial
        output_packet_features.append(create_packet(
            0.0,
            get_packet_size('handshake_initial_client', 1248, 30),
            0, 1, count_initial=1, count_crypto=1
        ))
        
        # 2. Server Initial + Handshake
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 's2c'),
            get_packet_size('handshake_initial_client', 1248, 50),
            1, 1,
            count_initial=1,
            count_handshake=np.random.randint(0, 2),
            count_ack=1,
            count_crypto=np.random.randint(1, 3)
        ))
        
        # 3. Possible Server Handshake continuation
        if np.random.random() > 0.4:
            output_packet_features.append(create_packet(
                get_delta_time('handshake', 's2c'),
                get_packet_size('handshake_other_server', 517, 100),
                1, 1, count_handshake=1, count_crypto=1
            ))
        
        # 4. Client Handshake completion
        output_packet_features.append(create_packet(
            get_delta_time('handshake', 'c2s'),
            get_packet_size('handshake_other_client', 1398, 100),
            0, 1,
            count_initial=np.random.randint(0, 2),
            count_handshake=1,
            count_1rtt=np.random.randint(0, 2),
            count_ack=np.random.randint(1, 3),
            count_padding=np.random.randint(0, 2),
            count_new_connection_id=1,
            count_crypto=np.random.randint(0, 2)
        ))
    
    # =================================================================
    # PHASE 2: HTTP/3 INITIALIZATION
    # =================================================================
    # Server sends HANDSHAKE_DONE + initial HTTP/3 setup
    settings_size = np.random.randint(15, 25)  # Variable SETTINGS size
    output_packet_features.append(create_packet(
        get_delta_time('data', 's2c'),
        get_packet_size('pkt_size_uni_server', 556, 100),
        1, 0,
        count_1rtt=1,
        count_ack=np.random.randint(0, 2),
        count_new_connection_id=np.random.randint(0, 2),
        count_crypto=np.random.randint(0, 2),
        count_handshake_done=1,
        http3_stream_count=1,
        stream_length=settings_size,
        stream_type_count=1
    ))
    server_app_bytes_sent += settings_size
    
    # Server sends unidirectional control streams
    server_uni_count = blueprint.get('server_uni_streams_count', 4)
    for i in range(max(0, server_uni_count - 1)):
        stream_len = np.random.choice([1, 1, 1, 26, 72], p=[0.5, 0.2, 0.1, 0.1, 0.1])
        is_fin = (i >= server_uni_count - 2) or (np.random.random() > 0.7)
        
        base_size = get_packet_size('pkt_size_uni_server', 92, 20)
        size = base_size + stream_len if stream_len > 1 else base_size
        
        output_packet_features.append(create_packet(
            get_delta_time('data', 's2c'),
            size, 1, 0,
            count_1rtt=1,
            http3_stream_count=1,
            http3_fin_count=1 if is_fin else 0,
            stream_length=stream_len,
            stream_type_count=1
        ))
        server_app_bytes_sent += stream_len
    
    # Client ACK
    output_packet_features.append(create_packet(
        get_delta_time('ack'),
        get_packet_size('ack_client', 91, 10),
        0, 0, count_1rtt=1, count_ack=1
    ))
    
    # =================================================================
    # PHASE 3: PRE-MIGRATION PROBING (if applicable)
    # =================================================================
    if blueprint.get('migration_type', 'NONE') != 'NONE':
        # Optional pre-migration path validation
        if np.random.random() > 0.5:  # 50% chance of early probing
            output_packet_features.append(create_packet(
                get_delta_time('data', 'c2s'),
                get_packet_size('path_challenge', 1398, 50),
                0, 0, count_1rtt=1, count_padding=1, count_path_challenge=1
            ))
            
            output_packet_features.append(create_packet(
                get_delta_time('data', 's2c'),
                get_packet_size('path_response', 1441, 50),
                1, 0, count_1rtt=1
            ))
            
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_client', 100, 15),
                0, 0, count_1rtt=1, count_ack=1, count_path_response=1
            ))
            
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_server', 91, 10),
                1, 0, count_1rtt=1, count_ack=1
            ))
    
    # =================================================================
    # PHASE 4: CLIENT APPLICATION DATA
    # =================================================================
    # Client sends unidirectional streams (QPACK, etc.)
    client_uni_count = blueprint.get('client_uni_streams_count', 4)
    for i in range(client_uni_count):
        # Variable stream sizes
        if i == 0:
            stream_len = np.random.randint(15, 25)  # SETTINGS-like
        elif i >= client_uni_count - 2:
            stream_len = np.random.choice([26, 72], p=[0.6, 0.4])  # Larger final streams
        else:
            stream_len = np.random.randint(1, 5)  # Small control streams
        
        is_fin = (i >= client_uni_count - 2) or (np.random.random() > 0.6)
        base_size = get_packet_size('pkt_size_uni_client', 110, 30)
        size = base_size + (stream_len if stream_len > 10 else 0)
        
        output_packet_features.append(create_packet(
            get_delta_time('data', 'c2s'),
            size, 0, 0,
            count_1rtt=1,
            http3_stream_count=1,
            http3_fin_count=1 if is_fin else 0,
            stream_length=stream_len,
            stream_type_count=1
        ))
        client_app_bytes_sent += stream_len
    
    # Client sends bidirectional requests
    client_bidi_count = blueprint.get('client_bidi_streams_count', 1)
    avg_request_size = blueprint.get('avg_request_size', 100)
    
    for i in range(client_bidi_count):
        # Variable request sizes around the average
        request_bytes = int(np.random.normal(avg_request_size, avg_request_size * 0.3))
        request_bytes = max(50, request_bytes)
        
        # Maybe piggyback ACK
        has_ack = np.random.random() > 0.5
        
        output_packet_features.append(create_packet(
            get_delta_time('data', 'c2s' if i == 0 else 's2c'),
            get_packet_size('pkt_size_bidi_client', 211, 50) + (request_bytes // 10),
            1 if i > 0 else 0, 0,  # Alternate direction sometimes
            count_1rtt=1,
            count_ack=1 if has_ack else 0,
            http3_stream_count=1,
            http3_fin_count=1,
            stream_length=request_bytes,
            stream_type_count=1
        ))
        client_app_bytes_sent += request_bytes
    
    # =================================================================
    # PHASE 5: CONNECTION MIGRATION
    # =================================================================
    migration_type = blueprint.get('migration_type', 'NONE')
    if migration_type != 'NONE' and not has_migrated:
        has_migrated = True
        
        # Wait until migration time
        time_until_migration = blueprint.get('time_to_migration_msec', 50.0) - current_time_msec
        if time_until_migration > 5.0:
            wait_delta = max(0.001, (time_until_migration / 1000.0) * np.random.uniform(0.8, 1.2))
        else:
            wait_delta = get_delta_time('data', 'c2s')
        
        # 1. Client PATH_CHALLENGE on new path (RFC 9000: padded to >= 1200 bytes)
        output_packet_features.append(create_packet(
            wait_delta,
            get_packet_size('path_challenge', PATH_VALIDATION_MTU_MIN + 100, 100),
            0, 0, count_1rtt=1, count_path_challenge=1, count_padding=1
        ))
        
        # 2. Server PATH_RESPONSE + PATH_CHALLENGE (bidirectional validation)
        validation_rtt = blueprint.get('migration_validation_duration_msec', 20.0) / 1000.0
        half_rtt = (validation_rtt / 2.0) * np.random.uniform(0.8, 1.2)
        
        output_packet_features.append(create_packet(
            half_rtt,
            get_packet_size('path_response', PATH_VALIDATION_MTU_MIN + 150, 100),
            1, 0,
            count_1rtt=1,
            count_path_response=1,
            count_path_challenge=1,
            count_padding=1,
            count_ack=np.random.randint(0, 2)
        ))
        
        # 3. Client PATH_RESPONSE
        output_packet_features.append(create_packet(
            half_rtt,
            get_packet_size('path_response', PATH_VALIDATION_MTU_MIN + 100, 100),
            0, 0,
            count_1rtt=1,
            count_ack=1,
            count_path_response=1,
            count_padding=1
        ))
        
        # 4. Server ACK
        output_packet_features.append(create_packet(
            get_delta_time('ack'),
            get_packet_size('ack_server', 91, 10),
            1, 0, count_1rtt=1, count_ack=1
        ))
    
    # =================================================================
    # PHASE 6: POST-MIGRATION DATA
    # =================================================================
    total_server_bytes = blueprint.get('total_server_app_bytes', 0)
    remaining_server_bytes = total_server_bytes - server_app_bytes_sent
    
    if remaining_server_bytes > 20:
        # Send remaining data in variable-sized packets
        avg_response_size = blueprint.get('avg_response_size', 71)
        num_response_packets = max(1, int(remaining_server_bytes / avg_response_size))
        
        for i in range(num_response_packets):
            bytes_this_packet = min(
                int(np.random.normal(avg_response_size, avg_response_size * 0.4)),
                remaining_server_bytes
            )
            bytes_this_packet = max(10, bytes_this_packet)
            
            is_fin = (i == num_response_packets - 1) or (remaining_server_bytes <= bytes_this_packet)
            
            output_packet_features.append(create_packet(
                get_delta_time('data', 's2c'),
                get_packet_size('pkt_size_bidi_server', 100, 40) + bytes_this_packet,
                1, 0,
                count_1rtt=1,
                http3_stream_count=1 if is_fin else 0,
                http3_fin_count=1 if is_fin else 0,
                stream_length=bytes_this_packet,
                stream_type_count=1 if is_fin else 0
            ))
            
            server_app_bytes_sent += bytes_this_packet
            remaining_server_bytes -= bytes_this_packet
            
            if remaining_server_bytes <= 0:
                break
    
    # =================================================================
    # PHASE 7: CONNECTION CLOSE
    # =================================================================
    close_type = blueprint.get('connection_close_type', 'CLIENT_CLOSE')
    
    # Wait until connection duration
    time_until_close = blueprint.get('connection_duration_msec', 100.0) - current_time_msec
    if time_until_close > 10.0:
        close_delta = max(0.010, (time_until_close / 1000.0) * np.random.uniform(0.9, 1.1))
    else:
        close_delta = get_delta_time('data', 'c2s')
    
    if close_type == 'CLIENT_CLOSE':
        output_packet_features.append(create_packet(
            close_delta,
            get_packet_size('close_client', 97, 15),
            0, 0, count_1rtt=1, count_connection_close=1
        ))
        
        # Server may ACK (not always captured)
        if np.random.random() > 0.3:
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_server', 91, 10),
                1, 0, count_1rtt=1, count_ack=1
            ))
    
    elif close_type == 'SERVER_CLOSE':
        output_packet_features.append(create_packet(
            close_delta,
            get_packet_size('close_server', 97, 15),
            1, 0, count_1rtt=1, count_connection_close=1
        ))
        
        if np.random.random() > 0.3:
            output_packet_features.append(create_packet(
                get_delta_time('ack'),
                get_packet_size('ack_client', 91, 10),
                0, 0, count_1rtt=1, count_ack=1
            ))
    
    return output_packet_features


# Example usage
if __name__ == "__main__":
    real_world_blueprint = {
        "connection_duration_msec": 107.25, 
        "retry_occurred": 1, 
        "server_issued_cid_count": 1, 
        "migration_type": "IP_AND_PORT", 
        "connection_close_type": "CLIENT_CLOSE",
        "handshake_duration_msec": 77.66, 
        "total_client_app_bytes": 119, 
        "total_server_app_bytes": 355, 
        "avg_request_size": 23.8, 
        "avg_response_size": 71.0,
        "client_bidi_streams_count": 1, 
        "client_uni_streams_count": 4, 
        "server_uni_streams_count": 4, 
        "time_to_migration_msec": 86.83, 
        "app_data_bytes_before_migration": 47, 
        "migration_validation_duration_msec": 1.17
    }
    
    example_stats = {
        'packet_sizes': {
            'handshake_initial_client': {'mean': 1248, 'std': 10, 'samples': 100},
            'handshake_other_server': {'mean': 517, 'std': 100, 'samples': 50},
            'handshake_other_client': {'mean': 1398, 'std': 50, 'samples': 50},
            'pkt_size_uni_server': {'mean': 100, 'std': 30, 'samples': 200},
            'pkt_size_uni_client': {'mean': 110, 'std': 25, 'samples': 200},
            'pkt_size_bidi_client': {'mean': 211, 'std': 40, 'samples': 100},
            'pkt_size_bidi_server': {'mean': 150, 'std': 50, 'samples': 200},
            'ack_client': {'mean': 91, 'std': 8, 'samples': 500},
            'ack_server': {'mean': 91, 'std': 8, 'samples': 500},
            'path_challenge': {'mean': 1398, 'std': 30, 'samples': 50},
            'path_response': {'mean': 1398, 'std': 30, 'samples': 50},
            'close_client': {'mean': 97, 'std': 10, 'samples': 50},
        },
        'delta_times': {
            'handshake_s2c': {'mean': 0.0010, 'std': 0.0008, 'samples': 100},
            'handshake_c2s': {'mean': 0.0020, 'std': 0.0015, 'samples': 100},
            'server_response': {'mean': 0.0003, 'std': 0.0002, 'samples': 1000},
            'client_request': {'mean': 0.0010, 'std': 0.0008, 'samples': 100},
            'ack_response': {'mean': 0.0003, 'std': 0.0002, 'samples': 500}
        }
    }
    
    # Generate 3 different captures from the same blueprint
    print("Generating 3 varied captures from the same blueprint...\n")
    
    for run in range(3):
        packets = generate_statistically_realistic_features(real_world_blueprint, example_stats)
        
        print(f"=== RUN {run + 1} ===")
        print(f"Generated {len(packets)} packets")
        print(f"Sample packets:")
        for pkt in packets[:3]:
            print(f"  #{pkt['frame_number']}: Δ{pkt['delta_time']:.6f}s, {pkt['packet_length']}B, dir={pkt['packet_direction']}")
        print(f"  ... (middle packets)")
        for pkt in packets[-2:]:
            print(f"  #{pkt['frame_number']}: Δ{pkt['delta_time']:.6f}s, {pkt['packet_length']}B, dir={pkt['packet_direction']}")
        print()
        
        # Save to CSV
        df = pd.DataFrame(packets)
        filename = f'generated_capture_run{run + 1}.csv'
        df.to_csv(filename, index=False)
        print(f"Saved to {filename}\n")

Generating 3 varied captures from the same blueprint...

=== RUN 1 ===
Generated 28 packets
Sample packets:
  #1: Δ0.000000s, 1247B, dir=0
  #2: Δ0.001086s, 94B, dir=1
  #3: Δ0.001564s, 1248B, dir=0
  ... (middle packets)
  #27: Δ0.097225s, 109B, dir=0
  #28: Δ0.000321s, 102B, dir=1

Saved to generated_capture_run1.csv

=== RUN 2 ===
Generated 28 packets
Sample packets:
  #1: Δ0.000000s, 1237B, dir=0
  #2: Δ0.000240s, 87B, dir=1
  #3: Δ0.001548s, 1230B, dir=0
  ... (middle packets)
  #27: Δ0.115783s, 96B, dir=0
  #28: Δ0.000004s, 81B, dir=1

Saved to generated_capture_run2.csv

=== RUN 3 ===
Generated 26 packets
Sample packets:
  #1: Δ0.000000s, 1254B, dir=0
  #2: Δ0.000139s, 104B, dir=1
  #3: Δ0.000673s, 1237B, dir=0
  ... (middle packets)
  #25: Δ0.000556s, 186B, dir=1
  #26: Δ0.103536s, 99B, dir=0

Saved to generated_capture_run3.csv



In [ ]:
real_world_blueprint = {
        "connection_duration_msec": 42.25, "retry_occurred": 1, "server_issued_cid_count": 1, "migration_type": "IP_AND_PORT", "connection_close_type": "CLIENT_CLOSE",
        "handshake_duration_msec": 12.66, "total_client_app_bytes": 119, "total_server_app_bytes": 355, "avg_request_size": 23.8, "avg_response_size": 71.0,
        "client_bidi_streams_count": 1, "client_uni_streams_count": 4, "server_uni_streams_count": 4, "time_to_migration_msec": 86.83, "app_data_bytes_before_migration": 47, "migration_validation_duration_msec": 1.17
    }

print("--- Generating features with STATISTICALLY REALISTIC script (Byte Totals Guaranteed) ---")
low_level_packets = generate_statistically_realistic_features(real_world_blueprint, full_stats_profile)
df = pd.DataFrame(low_level_packets)

display(df)

--- Generating features with STATISTICALLY REALISTIC script (Byte Totals Guaranteed) ---


,frame_number,delta_time,packet_length,packet_direction,header_form,count_initial,count_0rtt,count_handshake,count_1rtt,count_retry,...,count_path_challenge,count_path_response,count_new_connection_id,count_retire_cid,count_crypto,count_handshake_done,http3_stream_count,http3_fin_count,stream_length,stream_type_count
0,1,0.000000,1240,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2,0.003557,66,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,3,0.001530,1265,0,1,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,4,0.001694,60,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,5,0.000001,1211,0,1,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
5,6,0.000212,1324,1,1,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
6,7,0.005646,63,1,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
7,8,0.002740,1065,0,1,1,0,1,0,0,...,0,0,1,0,1,0,0,0,0,0
8,9,0.004113,279,1,0,0,0,0,1,0,...,0,0,0,0,0,1,1,0,24,1
9,10,0.012239,357,1,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,1,1


## Version 2

In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict
from typing import Dict, Any

In [2]:
class TrafficProfiler:
    def __init__(self, high_level_df: pd.DataFrame, low_level_df: pd.DataFrame):
        self.hl_df = high_level_df.copy()
        self.ll_df = low_level_df.copy()
        self.stat_profiles = {}

    def build_profiles(self) -> Dict[str, Any]:
        """Build statistical profiles from high-level and low-level dataframes."""
        # to know which low level dataset is which implementation
        merged = pd.merge(
            self.ll_df, 
            self.hl_df[['file_id', 'implementation']], 
            left_on='capture_id',
            right_on='file_id', 
            how='inner'
        )
        # get the unique implementations
        implementations = merged['implementation'].unique()

        for impl in implementations:
            print(f"\nBuilding detailed profile for implementation: {impl}")
            df = merged[merged['implementation'] == impl].copy()

            raw_stats = {
                'packet_sizes': defaultdict(list),
                'delta_times': defaultdict(list),
                'ack_frequency': defaultdict(list),
                'behavior': {}
            }

            # Direction: 0 = Client->Server, 1 = Server->Client
            is_client = df['packet_direction'] == 0
            is_server = df['packet_direction'] == 1
            is_long_header = df['header_form'] == 1 

            # ------ Handshake statistics ------
            hs_df = df[is_long_header] # packets with long header are found at the start of the connection, Initial and Handshake packets

            #Initial packets
            mask_initial_client = (hs_df['packet_direction'] == 0) & (hs_df['count_initial'] > 0)
            raw_stats['packet_sizes']['handshake_initial_client'] += hs_df.loc[
                mask_initial_client, 'packet_length'
            ].tolist()
            print(f"Collected {raw_stats['packet_sizes']['handshake_initial_client']} client initial handshake packet sizes.")

            mask_initial_server = (hs_df['packet_direction'] == 1) & (hs_df['count_initial'] > 0)
            raw_stats['packet_sizes']['handshake_initial_server'] = hs_df.loc[
                mask_initial_server, 'packet_length'
            ].tolist()

            #Other handshake packets
            raw_stats['packet_sizes']['handshake_other_server'] = hs_df.loc[
                (hs_df['packet_direction'] == 1) & (hs_df['count_handshake'] > 0), 'packet_length'
            ].tolist()
            
            raw_stats['packet_sizes']['handshake_other_client'] = hs_df.loc[
                (hs_df['packet_direction'] == 0) & (hs_df['count_handshake'] > 0), 'packet_length'
            ].tolist()

            raw_stats['delta_times']['handshake_c2s'] = hs_df.loc[hs_df['packet_direction'] == 0, 'delta_time'].tolist()
            raw_stats['delta_times']['handshake_s2c'] = hs_df.loc[hs_df['packet_direction'] == 1, 'delta_time'].tolist()

            # ------ 1RTT statistics ------
            rtt_df = df[~is_long_header].copy()

            # Migration behavior
            mask_pc = rtt_df['count_path_challenge'] > 0
            raw_stats['packet_sizes']['path_challenge'] = rtt_df.loc[mask_pc, 'packet_length'].tolist()
            
            mask_pr = rtt_df['count_path_response'] > 0
            raw_stats['packet_sizes']['path_response'] = rtt_df.loc[mask_pr, 'packet_length'].tolist()
            
            # some implementations add padding to path validation and path response packets, some not
            avg_pc_size = rtt_df.loc[mask_pc, 'packet_length'].mean() if len(rtt_df.loc[mask_pc]) > 0 else 0
            raw_stats['behavior']['padded_validation'] = avg_pc_size > 1000

            # packets with only ACK frames - no stream data or migration or connection close frames
            mask_ack_only = (
                (rtt_df['count_ack'] > 0) &
                (rtt_df['count_path_challenge'] == 0) &
                (rtt_df['count_path_response'] == 0) &
                (rtt_df['count_connection_close'] == 0) &
                (rtt_df['http3_stream_count'] == 0)
            )

            raw_stats['packet_sizes']['ack_client'] = rtt_df.loc[mask_ack_only & is_client, 'packet_length'].tolist()
            raw_stats['packet_sizes']['ack_server'] = rtt_df.loc[mask_ack_only & is_server, 'packet_length'].tolist()
            raw_stats['delta_times']['ack_response'] = rtt_df.loc[mask_ack_only, 'delta_time'].tolist()

            # Stream data statistics
            mask_stream = rtt_df['stream_length'] > 0

            # when a client sends stream data that is a request for the data

            raw_stats['packet_sizes']['pkt_size_bidi_client'] = rtt_df.loc[mask_stream & is_client, 'packet_length'].tolist()
            raw_stats['delta_times']['client_request'] = rtt_df.loc[mask_stream & is_client, 'delta_time'].tolist()

            # when a server sends stream data that is a response to the client - sending the downloaded file
            raw_stats['packet_sizes']['pkt_size_bidi_server'] = rtt_df.loc[mask_stream & is_server, 'packet_length'].tolist()
            raw_stats['delta_times']['server_response'] = rtt_df.loc[mask_stream & is_server, 'delta_time'].tolist()

            # Connection close behavior
            raw_stats['packet_sizes']['close_client'] = rtt_df.loc[
                (rtt_df['count_connection_close'] > 0) & is_client, 'packet_length'
            ].tolist()

            # ------- ACK -------

            grouped = df.groupby('file_id')
            for file_id, group in grouped:
                g = group.sort_values(by='frame_number')

                # after how many packets does the client send an ACK
                client_ack_indices = g[
                    (g['packet_direction'] == 0) & (g['count_ack'] > 0)
                ].index
                last_idx = g.index[0]
                for idx in client_ack_indices:
                    interval = g.loc[last_idx:idx]
                    server_pkts_count = len(interval[ (interval['packet_direction'] == 1) & (interval['stream_length'] > 0) ])
                    if server_pkts_count > 0:
                        raw_stats['ack_frequency']['client_sends_ack_after'].append(server_pkts_count)
                    last_idx = idx

                # server ACK behavior
                server_ack_indices = g[ (g['packet_direction'] == 1) & (g['count_ack'] > 0) ].index
                last_idx = g.index[0]
                
                for idx in server_ack_indices:
                    interval = g.loc[last_idx:idx]
                    client_pkts_count = len(interval[ (interval['packet_direction'] == 0) & (interval['stream_length'] > 0) ])
                    if client_pkts_count > 0:
                        raw_stats['ack_frequency']['server_sends_ack_after'].append(client_pkts_count)
                    last_idx = idx

            self.stat_profiles[impl] = self._compute_aggregates(raw_stats)
            
        return self.stat_profiles
            

    def _compute_aggregates(self, raw_stats):
        final_stats = defaultdict(dict)
        final_stats['behavior'] = raw_stats['behavior']
        for category in ['packet_sizes', 'delta_times', 'ack_frequency']:
                for key, data in raw_stats[category].items():
                    # Filter None or NaNs
                    clean_data = [x for x in data if pd.notna(x) and x > 0]
                    
                    if clean_data:
                        final_stats[category][key] = {
                            'mean': float(np.mean(clean_data)),
                            'std': float(np.std(clean_data)),
                            'min': float(np.min(clean_data)),
                            'max': float(np.max(clean_data)),
                            'samples': len(clean_data)
                        }
                    else:
                        # Fallback defaults to prevent crashes
                        final_stats[category][key] = {'mean': 100, 'std': 0, 'min': 100, 'max': 100, 'samples': 0}
        return dict(final_stats)
        

In [3]:
class SyntheticCaptureGenerator:
    def __init__(self, stats_profile):
        self.stats = stats_profile
        self.current_time = 0.0
        self.frame_number = 0
        # State variables
        self.client_ip = ""
        self.server_ip = ""
        self.client_port = 0
        self.server_port = 0
        self.is_migrated = False
        self.output_rows = []

        self.unacked_server_packets = 0
        self.unacked_client_packets = 0
        self.client_ack_threshold = 2 # Default fallback
        self.server_ack_threshold = 2 # Default fallback

    def generate(
        self, 
        blueprint: Dict[str, Any]
    ) -> pd.DataFrame:
        """
        Generate low-level packet features based on a high-level blueprint
        and statistical profiles.
        """
        self._reset_state(blueprint)
        
        impl = blueprint.get('implementation', 'quiche')
        profile = self.stats.get(impl, None)

        if not profile:
            # Try to grab the first available profile or raise error
            if self.stats:
                profile = list(self.stats.values())[0]
            else:
                raise ValueError(f"No statistical profiles available. Cannot generate traffic for {impl}")

        self._generate_handshake(blueprint, profile)

        total_client_bytes = blueprint.get('total_client_app_bytes', 0)
        total_server_bytes = blueprint.get('total_server_app_bytes', 0)

        #when to migrate, based on blueprint
        migration_type = blueprint.get('migration_type', 'BEFORE_DOWNLOAD')
        time_to_migration = blueprint.get('time_to_migration_msec', 0.5)

        if impl == 'quiche' and migration_type != 'NONE':
            # Quiche: Handshake -> Migration -> Data
            self._generate_migration(blueprint, profile)
            self._generate_data_transfer(total_client_bytes, total_server_bytes, profile)
            
        elif impl == 'aioquic' and migration_type != 'NONE':
            # Aioquic: Data -> Migration -> Data
            split_pct = 0.3 # Simulate migration partway through
            c_bytes_1 = int(total_client_bytes * split_pct)
            s_bytes_1 = int(total_server_bytes * split_pct)
            
            self._generate_data_transfer(c_bytes_1, s_bytes_1, profile)
            self._generate_migration(blueprint, profile)
            self._generate_data_transfer(total_client_bytes - c_bytes_1, total_server_bytes - s_bytes_1, profile)
            
        else:
            # Standard Data Transfer (No Migration)
            self._generate_data_transfer(total_client_bytes, total_server_bytes, profile)

        # --- PHASE 3: CLOSING ---
        self._generate_close(blueprint, profile)
        
        return pd.DataFrame(self.output_rows)


    def _reset_state(self, blueprint):
        self.current_time = float(blueprint.get('time_first', 0.0))
        self.frame_number = 0
        self.output_rows = []
        self.client_ip = blueprint['initial_ip_client']
        self.server_ip = blueprint['initial_ip_server']
        self.client_port = blueprint['initial_port_client']
        self.server_port = blueprint['initial_port_server']
        self.is_migrated = False
        self.unacked_client_packets = 0
        self.unacked_server_packets = 0


    def _sample_val(self, profile, category, key, default_mean=100, default_std=0):
        try:
            stats = profile[category].get(key)
            if stats and stats['samples'] > 0:
                val = np.random.normal(stats['mean'], stats['std'])
                return max(0.000001, val)
        except (KeyError, TypeError):
            pass
        return np.random.normal(default_mean, default_std)
    
    def _get_delta_time(self, profile, phase, direction):
        if(phase == 'handshake'):
            key = 'handshake_c2s' if direction == 0 else 'handshake_s2c'
            return self._sample_val(profile, 'delta_times', key, 0.002, 0.001)
        elif(phase == 'data'):
            key = 'client_request' if direction == 0 else 'server_response'
            default = 0.010 if direction == 0 else 0.001
            return self._sample_val(profile, 'delta_times', key, default, default/2)
        elif(phase == 'ack'):
            return self._sample_val(profile, 'delta_times', 'ack_response', 0.005, 0.002)
        return 0.001
    
    def _get_packet_size(self, profile, category, fallback=100):
        val = self._sample_val(profile, 'packet_sizes', category, fallback, fallback/10)
        return int(max(60, min(1350, val))) #trying to use realistic ethernet sizes
    
    def _add_packet(self, length, direction, flags, delta_time):
        self.frame_number += 1
        self.current_time += delta_time
        
        row = {
            'frame_number': self.frame_number,
            'delta_time': delta_time,
            'packet_length': length,
            'packet_direction': direction,
            'header_form': 1 if flags.get('count_initial') or flags.get('count_handshake') else 0,
            # Initialize all counts to 0
            'count_initial': 0, 'count_0rtt': 0, 'count_handshake': 0, 'count_1rtt': 0,
            'count_retry': 0, 'count_vn': 0, 'count_ack': 0, 'count_padding': 0,
            'count_connection_close': 0, 'count_path_challenge': 0, 'count_path_response': 0,
            'count_new_connection_id': 0, 'count_retire_cid': 0, 'count_ping': 0,
            'count_crypto': 0, 'count_handshake_done': 0, 'http3_stream_count': 0,
            'http3_fin_count': 0, 'stream_length': 0, 'stream_type_count': 0
        }
        
        row.update(flags)
        
        # Auto-set 1-RTT if it's a short header packet
        if row['header_form'] == 0:
            row['count_1rtt'] = 1
            
        self.output_rows.append(row)

    # --- GENERATION LOGIC ---

    def _generate_handshake(self, blueprint, profile):
        # 1. Client Initial
        dt = 0.0 # First packet
        size = self._get_packet_size(profile, 'handshake_initial_client', 1250)
        self._add_packet(size, 0, {'count_initial': 1, 'count_crypto': 1}, dt)
        
        # 2. Server Initial
        dt = self._get_delta_time(profile, 'handshake', 1)
        size = self._get_packet_size(profile, 'handshake_initial_server', 1250)
        self._add_packet(size, 1, {'count_initial': 1, 'count_crypto': 1, 'count_ack': 1}, dt)
        
        # 3. Server Handshake
        dt = self._get_delta_time(profile, 'handshake', 1) # Usually bursty/close to Initial
        size = self._get_packet_size(profile, 'handshake_other_server', 1200)
        self._add_packet(size, 1, {'count_handshake': 1, 'count_crypto': 1}, dt)
        
        # 4. Client Handshake + ACK
        dt = self._get_delta_time(profile, 'handshake', 0)
        size = self._get_packet_size(profile, 'handshake_other_client', 1200)
        self._add_packet(size, 0, {'count_handshake': 1, 'count_crypto': 1, 'count_ack': 1}, dt)

        # 5. Handshake Done (Server)
        dt = self._get_delta_time(profile, 'data', 1)
        self._add_packet(600, 1, {'count_handshake_done': 1, 'count_1rtt': 1, 'count_ack': 1}, dt)

    def _generate_migration(self, blueprint, profile):
        # 1. Path Challenge (Client)
        # Check if this implementation uses padding
        is_padded = profile['behavior'].get('padded_validation', True)
        
        # Use explicit blueprint time if available, or fall back to profile/default
        jump_time = blueprint.get('time_to_migration_msec', 0.0) 
        if jump_time > 0:
             # If blueprint specifies absolute time, we calculate delta
             # But blueprint is usually "time TO migration", which implies offset from start
             # Simplified: just use a larger delta to simulate the gap
             dt = max(0.01, float(jump_time) / 1000.0)
        else:
             dt = 0.05

        size_key = 'path_challenge' if not is_padded else 'handshake_initial_client' # Fallback for large size
        size = self._get_packet_size(profile, size_key, 1250 if is_padded else 64)
        
        self._add_packet(size, 0, {
            'count_path_challenge': 1, 
            'count_padding': 1 if is_padded else 0
        }, dt)
        
        # 2. Path Response (Server)
        latency = blueprint.get('first_path_validation_response_latency', 0.02)
        size = self._get_packet_size(profile, 'path_response', size)
        self._add_packet(size, 1, {'count_path_response': 1}, latency)
        
        self.is_migrated = True

    def _generate_data_transfer(self, client_bytes, server_bytes, profile):
        """
        Sophisticated data generation that respects ACK frequency stats.
        """
        mss = 1350
        
        # Initialize ACK thresholds from profile
        self.client_ack_threshold = int(self._sample_val(profile, 'ack_frequency', 'client_sends_ack_after', 2, 1))
        self.server_ack_threshold = int(self._sample_val(profile, 'ack_frequency', 'server_sends_ack_after', 2, 1))

        while client_bytes > 0 or server_bytes > 0:
            
            # --- CLIENT SENDING ---
            if client_bytes > 0:
                chunk = min(client_bytes, mss)
                dt = self._get_delta_time(profile, 'data', 0)
                size = self._get_packet_size(profile, 'pkt_size_bidi_client', chunk + 50)
                
                self._add_packet(size, 0, {'http3_stream_count': 1, 'stream_length': chunk}, dt)
                client_bytes -= chunk
                
                # Track for Server ACK logic
                self.unacked_client_packets += 1
                if self.unacked_client_packets >= self.server_ack_threshold:
                    # Server sends ACK
                    ack_dt = self._get_delta_time(profile, 'ack', 1)
                    ack_size = self._get_packet_size(profile, 'ack_server', 90)
                    self._add_packet(ack_size, 1, {'count_ack': 1}, ack_dt)
                    
                    self.unacked_client_packets = 0
                    self.server_ack_threshold = int(self._sample_val(profile, 'ack_frequency', 'server_sends_ack_after', 2, 1))

            # --- SERVER SENDING (Burst Logic) ---
            # Server usually sends more data (Download). Send a small burst.
            burst_size = np.random.randint(1, 5) 
            for _ in range(burst_size):
                if server_bytes <= 0: break
                
                chunk = min(server_bytes, mss)
                dt = self._get_delta_time(profile, 'data', 1)
                size = self._get_packet_size(profile, 'pkt_size_bidi_server', chunk + 50)
                
                self._add_packet(size, 1, {'http3_stream_count': 1, 'stream_length': chunk}, dt)
                server_bytes -= chunk
                
                # Track for Client ACK logic
                self.unacked_server_packets += 1
                if self.unacked_server_packets >= self.client_ack_threshold:
                    # Client sends ACK
                    ack_dt = self._get_delta_time(profile, 'ack', 0)
                    ack_size = self._get_packet_size(profile, 'ack_client', 90)
                    self._add_packet(ack_size, 0, {'count_ack': 1}, ack_dt)
                    
                    self.unacked_server_packets = 0
                    self.client_ack_threshold = int(self._sample_val(profile, 'ack_frequency', 'client_sends_ack_after', 2, 1))

    def _generate_close(self, bp, profile):
        closer = bp.get('connection_close_type', 'CLIENT_CLOSE')
        direction = 0 if closer == 'CLIENT_CLOSE' else 1
        
        dt = self._get_delta_time(profile, 'data', direction)
        size_key = 'close_client' if direction == 0 else 'close_server'
        size = self._get_packet_size(profile, size_key, 250)
        
        self._add_packet(size, direction, {'count_connection_close': 1}, dt)


In [4]:
hl_df = pd.read_csv(r'../../high_level_features\dataset\all_captures_dataset.csv')

In [5]:
hl_df

,file_id,implementation,initial_ip_client,initial_ip_server,initial_port_client,initial_port_server,time_first,time_last,connection_duration,version_negotiation_occurred,...,ack_sent_client,ack_sent_server,crypto_sent_client,crypto_sent_server,handshake_done_client,handshake_done_server,path_challenge_sent_client,path_challenge_sent_server,path_response_sent_client,path_response_sent_server
0,3172,aioquic,127.0.0.2,127.0.0.1,49668,4433,2025-11-17T17-47-37.011112,2025-11-17T17-47-37.060358,49.246073,0,...,5,3,2,3,0,1,0,1,1,0
1,3173,aioquic,127.0.0.2,127.0.0.1,50461,4433,2025-11-17T17-47-40.410962,2025-11-17T17-47-40.433849,22.886992,0,...,4,2,2,3,0,1,0,1,1,0
2,3174,aioquic,127.0.0.2,127.0.0.1,49669,4433,2025-11-17T17-47-43.799772,2025-11-17T17-47-43.824725,24.952888,0,...,4,2,2,3,0,1,0,1,1,0
3,3175,aioquic,127.0.0.2,127.0.0.1,55759,4433,2025-11-17T17-47-47.151185,2025-11-17T17-47-47.182836,31.651020,0,...,5,2,2,3,0,1,0,1,1,0
4,3176,aioquic,127.0.0.2,127.0.0.1,49669,4433,2025-11-17T17-47-50.616860,2025-11-17T17-47-50.652164,35.304070,0,...,5,3,2,3,0,1,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11906,997,quiche,127.0.0.2,127.0.0.1,57185,4433,2025-11-03T16-56-54.481458,2025-11-03T16-56-54.495120,13.662100,1,...,5,5,3,4,0,1,1,1,1,1
11907,998,quiche,127.0.0.2,127.0.0.1,63694,4433,2025-11-03T16-56-57.130233,2025-11-03T16-56-57.145495,15.261889,1,...,5,5,3,4,0,1,1,1,1,1
11908,999,quiche,127.0.0.2,127.0.0.1,52678,4433,2025-11-03T16-56-59.698150,2025-11-03T16-56-59.709128,10.977983,1,...,4,5,3,4,0,1,1,1,1,1
11909,99,quiche,127.0.0.2,127.0.0.1,49666,4433,2025-11-03T15-53-32.850111,2025-11-03T15-53-32.862704,12.593031,1,...,5,5,3,4,0,1,1,1,1,1


In [6]:
ll_df = pd.read_csv(r'../../low_level_features\dataset\all_low_level_features_merged.csv')

In [7]:
ll_df

,frame_number,delta_time,packet_length,packet_direction,header_form,count_initial,count_0rtt,count_handshake,count_1rtt,count_retry,...,count_new_connection_id,count_retire_cid,count_ping,count_crypto,count_handshake_done,http3_stream_count,http3_fin_count,stream_length,stream_type_count,capture_id
0,1,0.000000,1242,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,10000
1,2,0.000087,89,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,10000
2,3,0.000554,1242,0,1,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,10000
3,4,0.001087,1242,1,1,1,0,1,0,0,...,0,0,0,4,0,0,0,0,0,10000
4,5,0.000021,403,1,1,0,0,1,0,0,...,0,0,0,2,0,0,0,0,0,10000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
261191,23,0.000215,75,1,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,9
261192,24,0.000054,147,0,0,0,0,0,1,0,...,0,0,0,0,0,1,1,72,1,9
261193,25,0.000193,101,0,0,0,0,0,1,0,...,0,0,0,0,0,1,1,26,1,9
261194,26,0.001212,310,1,0,0,0,0,1,0,...,0,0,0,0,0,1,1,230,1,9


In [8]:
profiler = TrafficProfiler(hl_df, ll_df)
stats_profiles = profiler.build_profiles()


Building detailed profile for implementation: nginx
Collected [1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 1242, 1392, 

In [9]:
generator = SyntheticCaptureGenerator(stats_profiles)

In [10]:
# hl_df['time_first'] = (hl_df['time_first'].str.replace(r'T(\d+)-(\d+)-', r'T\1:\2:', regex=True).pipe(pd.to_datetime))
hl_df['time_first'] = 0.0

In [11]:
blueprint = hl_df.iloc[10000].to_dict() 

In [12]:
display(hl_df.iloc[10000]['total_server_app_bytes'])

277

In [13]:
blueprint

{'file_id': 2134,
 'implementation': 'quiche',
 'initial_ip_client': '127.0.0.2',
 'initial_ip_server': '127.0.0.1',
 'initial_port_client': 64982,
 'initial_port_server': 4433,
 'time_first': 0.0,
 'time_last': '2025-11-03T17-46-17.995523',
 'connection_duration': 10.96796989440918,
 'version_negotiation_occurred': 1,
 'retry_occurred': 1,
 'new_connection_ids_issued_server': 1,
 'retired_cid_count_client': 0,
 'retired_cid_count_server': 0,
 'new_connection_ids_issued_client': 1,
 'migration_type': 'BEFORE_DOWNLOAD',
 'first_path_validation_response_latency': 0.4708766937255859,
 'path_validation_initiated': 1.0,
 'connection_close_type': 'CLIENT_CLOSE',
 'bytes_sent_client': 7194,
 'bytes_sent_server': 4568,
 'packets_sent_client': 13,
 'packets_sent_server': 12,
 'quic_packets_sent_client': 18,
 'quic_packets_sent_server': 19,
 'handshake_duration': 4.890203475952148,
 'time_to_migration': 6.298065185546875,
 'migration_duration': 0.8909702301025391,
 'packets_before_migration': 13

In [14]:
synthetic_packets = generator.generate(blueprint)

In [15]:
synthetic_data = pd.DataFrame(synthetic_packets)

In [16]:
pd.set_option('display.max_columns', None)

In [17]:
synthetic_data

,frame_number,delta_time,packet_length,packet_direction,header_form,count_initial,count_0rtt,count_handshake,count_1rtt,count_retry,count_vn,count_ack,count_padding,count_connection_close,count_path_challenge,count_path_response,count_new_connection_id,count_retire_cid,count_ping,count_crypto,count_handshake_done,http3_stream_count,http3_fin_count,stream_length,stream_type_count
0,1,0.000000,1350,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,2,0.001553,1232,1,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
2,3,0.001903,475,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,4,0.001701,1350,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
4,5,0.001252,600,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
5,6,0.050000,1347,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0
6,7,0.470877,390,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
7,8,0.000342,87,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,119,0
8,9,0.000001,201,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,277,0
9,10,0.000001,81,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0


problems: not enough packets, not enough acks - basically after the handshake there are none, not inclusive enough for the other implementations, padding only in path challenge, no new connection id, no http fin, no stream type count

basically the script is not following the parts of the blueprint, like connection id issued, connection duration, time to migration.

In [18]:
import numpy as np
import pandas as pd
from typing import Dict, Any, List

class LinearSyntheticGenerator:
    def __init__(self, stats_profile: Dict[str, Any]):
        self.stats = stats_profile
        
        # Simulation State
        self.current_time = 0.0
        self.frame_number = 0
        self.output_rows = []
        self.network_jitter = 1.0
        
        # ACK Tracking
        self.packets_since_client_ack = 0
        self.packets_since_server_ack = 0

    def generate(self, blueprint: Dict[str, Any]) -> pd.DataFrame:
        self._reset_state(blueprint)
        
        # 1. Select Profile
        impl = blueprint.get('implementation', 'quiche')
        profile = self.stats.get(impl)
        if not profile:
            # Fallback to first available if specific impl missing
            profile = list(self.stats.values())[0] if self.stats else {}

        # 2. Phase 1: Handshake
        self._generate_handshake(blueprint, profile)
        
        # 3. Phase 2: HTTP/3 Setup (Control Streams)
        self._generate_control_streams(blueprint, profile)

        # 4. Phase 3: Data Transfer & Migration
        # We handle this differently based on implementation behavior
        total_server_bytes = int(blueprint.get('total_server_app_bytes', 1000))
        total_client_bytes = int(blueprint.get('total_client_app_bytes', 100))
        migration_type = blueprint.get('migration_type', 'NONE')

        if impl == 'aioquic' and migration_type != 'NONE':
            # AIOQUIC: Migration happens DURING the download
            # Strategy: Send Request -> Send 30% Response -> Migrate -> Send Rest
            
            # A. Client sends request
            self._generate_client_request(total_client_bytes, profile)
            
            # B. Server sends part of response
            bytes_before = int(total_server_bytes * 0.3)
            self._generate_server_response(bytes_before, profile)
            
            # C. Migration
            self._generate_migration(blueprint, profile)
            
            # D. Server sends rest
            self._generate_server_response(total_server_bytes - bytes_before, profile)

        elif impl == 'quiche' and migration_type != 'NONE':
            # QUICHE: Migration happens BEFORE the download starts
            
            # A. Migration
            self._generate_migration(blueprint, profile)
            
            # B. Client sends request
            self._generate_client_request(total_client_bytes, profile)
            
            # C. Server sends response
            self._generate_server_response(total_server_bytes, profile)

        else:
            # NO MIGRATION (Standard)
            self._generate_client_request(total_client_bytes, profile)
            self._generate_server_response(total_server_bytes, profile)

        # 5. Phase 4: Connection Close
        self._generate_close(blueprint, profile)
        
        return pd.DataFrame(self.output_rows)

    def _reset_state(self, bp):
        self.current_time = float(bp.get('time_first', 0.0))
        self.frame_number = 0
        self.output_rows = []
        self.packets_since_client_ack = 0
        self.packets_since_server_ack = 0
        self.network_jitter = np.random.uniform(0.8, 1.2) # Add randomness to this run

    # =========================================================================
    #  PHASE GENERATORS
    # =========================================================================

    def _generate_handshake(self, bp, profile):
        # 1. Client Initial
        self._add_packet(1250, 0, {'count_initial': 1, 'count_crypto': 1}, 0.0)
        
        # 2. Server Initial
        dt = self._get_delta(profile, 'handshake_s2c')
        self._add_packet(1250, 1, {'count_initial': 1, 'count_crypto': 1, 'count_ack': 1}, dt)
        
        # 3. Server Handshake
        dt = 0.001 # Burst
        self._add_packet(self._get_size(profile, 'handshake_other_server'), 1, {'count_handshake': 1, 'count_crypto': 1}, dt)
        
        # 4. Client Handshake + ACK
        dt = self._get_delta(profile, 'handshake_c2s')
        self._add_packet(self._get_size(profile, 'handshake_other_client'), 0, {'count_handshake': 1, 'count_crypto': 1, 'count_ack': 1}, dt)

        # 5. Handshake Done
        dt = 0.001
        self._add_packet(600, 1, {'count_handshake_done': 1, 'count_1rtt': 1, 'count_ack': 1}, dt)

        # 6. New CIDs (Server)
        count = int(bp.get('server_issued_cid_count', 1))
        for _ in range(count):
            self._add_packet(80, 1, {'count_new_connection_id': 1, 'count_1rtt': 1}, 0.0005)

    def _generate_control_streams(self, bp, profile):
        # Settings frames, etc.
        dt = 0.002
        # Client Settings
        self._add_packet(100, 0, {'http3_stream_count': 1, 'stream_length': 30, 'count_1rtt': 1}, dt)
        # Server Settings
        self._add_packet(100, 1, {'http3_stream_count': 1, 'stream_length': 30, 'count_1rtt': 1}, dt)
        
        # Force an ACK cycle after control streams
        self._send_ack(0, profile)

    def _generate_migration(self, bp, profile):
        # 1. Path Challenge (Client)
        is_padded = profile['behavior'].get('padded_validation', True)
        size = 1250 if is_padded else 64
        
        # Time to migration
        jump_time = bp.get('time_to_migration_msec', 50) / 1000.0
        # If jump_time is absolute (e.g. 10s), we calculate delta, otherwise use it as delta
        dt = max(0.01, jump_time if jump_time < 1.0 else 0.05) 
        
        self._add_packet(size, 0, {'count_path_challenge': 1, 'count_padding': 1 if is_padded else 0, 'count_1rtt': 1}, dt)
        
        # 2. Path Response (Server)
        latency = bp.get('first_path_validation_response_latency', 0.02)
        self._add_packet(size, 1, {'count_path_response': 1, 'count_1rtt': 1}, latency)
        
        # 3. Client confirms with ACK/Response
        self._add_packet(100, 0, {'count_path_response': 1, 'count_ack': 1, 'count_1rtt': 1}, 0.002)

    def _generate_client_request(self, total_bytes, profile):
        if total_bytes <= 0: return
        
        # Client requests are usually small (GET /index.html is ~50 bytes), so typically 1 packet
        dt = self._get_delta(profile, 'client_request')
        size = self._get_size(profile, 'pkt_size_bidi_client', fallback=total_bytes + 50)
        
        self._add_packet(size, 0, {
            'http3_stream_count': 1, 
            'stream_length': total_bytes, 
            'count_1rtt': 1,
            'http3_fin_count': 1 # Request usually sets FIN
        }, dt)
        
        self.packets_since_client_ack += 1

    def _generate_server_response(self, total_bytes, profile):
        if total_bytes <= 0: return

        # Determine packet size from profile
        avg_pkt_size = self._get_size(profile, 'pkt_size_bidi_server', 1350)
        mss = min(1350, avg_pkt_size)
        
        # Calculate exactly how many packets we need
        # e.g., 5000 bytes / 1350 = 3 packets (1350, 1350, 1350, 950)
        remaining = total_bytes
        
        ack_threshold = int(self._sample_val(profile, 'ack_frequency', 'client_sends_ack_after', 2))

        while remaining > 0:
            chunk = min(remaining, mss)
            
            dt = self._get_delta(profile, 'server_response')
            size = chunk + 35 # Headers/Overhead
            
            is_fin = (chunk == remaining)
            
            self._add_packet(size, 1, {
                'http3_stream_count': 1, 
                'stream_length': chunk, 
                'count_1rtt': 1,
                'http3_fin_count': 1 if is_fin else 0
            }, dt)
            
            remaining -= chunk
            self.packets_since_server_ack += 1
            
            # Check for ACK
            if self.packets_since_server_ack >= ack_threshold:
                self._send_ack(0, profile) # Client ACKs
                ack_threshold = int(self._sample_val(profile, 'ack_frequency', 'client_sends_ack_after', 2))

    def _generate_close(self, bp, profile):
        closer = bp.get('connection_close_type', 'CLIENT_CLOSE')
        direction = 0 if closer == 'CLIENT_CLOSE' else 1
        
        self._add_packet(250, direction, {'count_connection_close': 1, 'count_1rtt': 1}, 0.005)
        
        # Other side might ACK
        if np.random.random() > 0.5:
            self._send_ack(1 if direction == 0 else 0, profile)

    def _send_ack(self, direction, profile):
        dt = self._get_delta(profile, 'ack_response')
        size = self._get_size(profile, f'ack_{"client" if direction==0 else "server"}', 60)
        self._add_packet(size, direction, {'count_ack': 1, 'count_1rtt': 1}, dt)
        
        if direction == 0: self.packets_since_server_ack = 0
        else: self.packets_since_client_ack = 0

    # =========================================================================
    #  HELPERS (Interface to your Profiler)
    # =========================================================================

    def _add_packet(self, length, direction, flags, delta_time):
        self.frame_number += 1
        # Apply jitter
        dt = max(0.000001, delta_time * self.network_jitter)
        self.current_time += dt
        
        row = {
            'frame_number': self.frame_number,
            'delta_time': dt,
            'packet_length': int(length),
            'packet_direction': direction,
            'header_form': 1 if flags.get('count_initial') or flags.get('count_handshake') else 0,
            # Defaults
            'count_initial': 0, 'count_0rtt': 0, 'count_handshake': 0, 'count_1rtt': 0,
            'count_retry': 0, 'count_vn': 0, 'count_ack': 0, 'count_padding': 0,
            'count_connection_close': 0, 'count_path_challenge': 0, 'count_path_response': 0,
            'count_new_connection_id': 0, 'count_retire_cid': 0, 'count_ping': 0,
            'count_crypto': 0, 'count_handshake_done': 0, 
            'http3_stream_count': 0, 'http3_fin_count': 0, 
            'stream_length': 0, 'stream_type_count': 0
        }
        row.update(flags)
        
        # Track for ACKs (if this packet has content)
        if row['stream_length'] > 0 or row['count_crypto'] > 0:
            if direction == 0: self.packets_since_client_ack += 1
            else: self.packets_since_server_ack += 1
            
        self.output_rows.append(row)

    def _get_size(self, profile, key, fallback=100):
        try:
            stats = profile['packet_sizes'][key]
            # Simple normal distribution sampling
            val = int(np.random.normal(stats['mean'], stats['std']))
            return max(60, min(1350, val))
        except:
            return fallback

    def _get_delta(self, profile, key):
        try:
            stats = profile['delta_times'][key]
            val = np.random.normal(stats['mean'], stats['std'])
            return max(0.0001, val)
        except:
            return 0.005

    def _sample_val(self, profile, category, key, fallback):
        try:
            stats = profile[category][key]
            val = np.random.normal(stats['mean'], stats['std'])
            return max(1, val)
        except:
            return fallback

In [19]:
linear_generator = LinearSyntheticGenerator(stats_profiles)

In [23]:
synthetic_data_2 = linear_generator.generate(blueprint)

In [24]:
dataframe = pd.DataFrame(synthetic_data_2)

In [25]:
dataframe

,frame_number,delta_time,packet_length,packet_direction,header_form,count_initial,count_0rtt,count_handshake,count_1rtt,count_retry,count_vn,count_ack,count_padding,count_connection_close,count_path_challenge,count_path_response,count_new_connection_id,count_retire_cid,count_ping,count_crypto,count_handshake_done,http3_stream_count,http3_fin_count,stream_length,stream_type_count
0,1,0.000001,1250,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,2,0.001960,1250,1,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
2,3,0.000908,1350,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,4,0.000091,1350,0,1,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0
4,5,0.000908,600,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
5,6,0.000454,80,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
6,7,0.001816,100,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,30,0
7,8,0.001816,100,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,30,0
8,9,0.000156,75,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
9,10,0.045411,1250,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0
